# ISM / Confocal image processing for fluorescence lifetime reconstruction. TTM acquisition 

### Data Loading and Phasors calculation on ISM image's (Nx, Ny) pixels

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
while not (ROOT / "src").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import h5py
import matplotlib.pyplot as plt
import numpy as np

import brighteyes_flim.tools_phasor as brighteyes_flim
import brighteyes_ism.analysis.Graph_lib as gr
import brighteyes_ism.dataio.mcs as mcs


In [ ]:
with h5py.File(r"C:\Users\REPLACE_ME\Desktop\images\convallaria_03_07_2024.h5","a") as f:
    
     print(f.keys())
     data_input = f["dataset_1"]  

     brighteyes_flim.phasor_h5(data_path = r"C:\Users\REPLACE_ME\Desktop\images\convallaria_03_07_2024.h5", data_input = data_input) 

In [ ]:
# Display the pixels'histogram-associated phasors in the phasor plot

%matplotlib widget

hf_phasors_per_pixel = h5py.File(r"C:\Users\REPLACE_ME\Desktop\images\convallaria_03_07_2024_phasors_matrix.h5", "r")
print(hf_phasors_per_pixel.keys())

phasors_pix = hf_phasors_per_pixel["h5_dataset_phasor_pix"]  # data with phasors in each pixel
#fasors_pix[1:100, 1:100]
print(phasors_pix.shape)

brighteyes_flim.plot_phasor(phasors_pix[:], bins_2dplot=200, log_scale=False, quadrant='first', dfd_freq = 80e6)

### Sum over the time bin and channel dimensions to get the intensity (Nx x Ny) image

In [ ]:

with h5py.File(r"C:\Users\REPLACE_ME\Desktop\images\convallaria_03_07_2024.h5","a") as f:
    
        print(f.keys())
        data_input = f["dataset_1"]
        
        data_histograms = np.sum(data_input, axis = (2, 3))
        print(data_histograms.shape)
    
# Plot the histogram of the photon counts in each pixel to see the distribution (e.g. check the level of noise) 
        plt.figure()
        plt.hist(data_histograms.flatten(), bins = 100, range = (0, 500))

# Lifetime Analysis 

#### Calculate the fluorescence lifetime from the phasor for each pixel with the formula below (f = dfd frequency or laser rep rate frequency):
#### τφ = (1/(2πf)) * tan(φ)
#### φ = arctan(s/g)
#### g = Re{phasors_pix}
#### s = Im{phasors_pix}

In [ ]:
# Calculate the single pixels' tau (fluorescence lifetime) values from the phasors of each pixel 

tau_phi = brighteyes_flim.calculate_tau_phi(np.real(phasors_pix[:]), np.imag(phasors_pix[:]), dfd_freq = 80e6)
print(tau_phi.shape)


#### Calculate the fluorescence lifetime from the phasor for each pixel with the formula below (f = dfd frequency or laser rep rate frequency):
#### τ<sub>m</sub> = (1/2*π*f) * √(1/m<sup>2</sup> - 1)
#### m = √g<sup>2</sup> + s<sup>2</sup>
#### g = Re{phasor_corrected}
#### s = Im{phasor_corrected}

In [ ]:
tau_m = brighteyes_flim.calculate_tau_m(np.real(phasors_pix), np.imag(phasors_pix), dfd_freq = 80e6)
print(tau_m.shape)

### Visualize histograms of tau distribution in the pixels

In [ ]:
# Plot the distribution of tau values in all the pixels of the image 
tau_data = 1e9*tau_phi.flatten()

plt.figure()
plt.hist(tau_data, range = (-6, 10), bins = 50)

In [ ]:
tau_m_data = 1e9*tau_m.flatten()

plt.figure()
plt.hist(tau_m_data, range = (0, 13), bins = 50)

### Filter the image with a thresholding Otsu algorithm to select the pixels with a minimum number of photons

In [ ]:
# Process the histograms in each pixel to remove the pixels with a low amount of signal

# We want to process the histogram counts matrix and the tau matrix 
# to check if the pixels with high counts have only tau positives 

from skimage.filters import threshold_otsu as otsu
histograms_filtered = otsu(image=data_histograms, nbins=50)
print(histograms_filtered)
hist_indexes = np.argwhere(data_histograms > 70)
histogram_denoised = data_histograms[hist_indexes[:, 0], hist_indexes[:, 1]]
print(histogram_denoised.shape)
print(hist_indexes.shape)

In [ ]:
# Select the tau only from the pixels having a sufficient amount of photons, 
# identified applying an image thresholding method (e.g., Otsu's method).

tau_phi_denoised = tau_phi[hist_indexes[:, 0], hist_indexes[:, 1]]
print(tau_phi_denoised)
plt.figure()
plt.hist(1e9*tau_phi_denoised, bins = 100, range = (-1, 10))


In [ ]:
# Select the tau only from the pixels having a sufficient amount of photons, 
# identified applying an image thresholding method (e.g., Otsu's method).

tau_m_denoised = tau_m[hist_indexes[:, 0], hist_indexes[:, 1]]
print(tau_m_denoised.shape)
plt.figure()
plt.hist(1e9*tau_m_denoised, bins = 100, range = (-1, 10))


### Display and save the FLIM image representing the lifetime and intensity with a 2D colormap

In [ ]:
fig = plt.figure(figsize = (9, 6))
gs = fig.add_gridspec(4, 4)
ax1 = fig.add_subplot(gs[0:2, 0:2])
ax2 = fig.add_subplot(gs[2:4, 0:2])
brighteyes_flim.plot_phasor(phasors_pix[:], bins_2dplot=200, log_scale=False, quadrant='first', fig = fig, ax = ax1, dfd_freq = 80e6)
gr.show_flim(data_histograms, tau_m*1e9, pxsize = 0.16, pxdwelltime = 200, lifetime_bounds = (1.3, 3.6), intensity_bounds = (0, 250),
             fig = fig, ax = ax2)  
fig.tight_layout()
plt.savefig(r"C:\Users\REPLACE_ME\Desktop\PDF_processed_images\convallaria_03_07_2024_TTM.pdf", dpi = 900)